# Publishing a Tool to the tool registry
This notebook showcases how to use the tool-registry REST API to publish a tool.

In [ ]:
import requests
import json

In [ ]:
API_URL_BASE = "https://tool-registry-api.eosc-data-commons.dansdemo.nl/api/v1/"
API_URL_TOOLS = f"{API_URL_BASE}tools/"
API_URL_SOURCES = f"{API_URL_BASE}sources/"

TOOL_REGISTRY_VERSION = "1.0.4"

## Getting access credentials
Before publishing to the tool registry an EGI access token is needed. This can be obtained from [https://aai-dev.egi.eu/token/](https://aai-dev.egi.eu/token/). 
Once obtained copy and paste in the cell below. The token is only valid for 1 hour.

In [ ]:
TOKEN = ""

# Accepted Tool URLs

The Tool Registry accepts tools published at the following locations:

- **GitHub repository** – A complete repository containing the tool's source code and metadata.
- **GitHub repository subfolder** – A specific directory within a repository, allowing individual tools or notebooks to be registered separately.
- **GitLab repository** – A repository hosted on a GitLab instance containing the tool's source code and metadata.
- **Zenodo URL** – A published software record on Zenodo, preferably with a persistent DOI.

## Metadata Harvesting

During the harvesting process, the Tool Registry attempts to automatically discover and extract metadata from the submitted URL.

The harvester looks for the following metadata formats:

- **`codemeta.json`** – Structured software metadata following the [CodeMeta](https://codemeta.github.io/) specification. This is the preferred format for describing research software, including its purpose, authors, version, dependencies and other relevant properties.
- **RO-Crate (`ro-crate-metadata.json`)** – Metadata following the [RO-Crate](https://www.researchobject.org/ro-crate/) specification, describing research software, workflows, notebooks and their relationships to other research objects.
- **`pyproject.toml`** – Python project metadata, including the package name, description, version, authors and dependencies.


## Important Metadata Fields

To improve tool discovery and matching, particular attention should be given to the following metadata fields:

- **Name** – A clear, descriptive name identifying the tool.
- **Description** – A meaningful explanation of what the tool does, which problems it addresses and what types of data it processes. The description should mention the scientific or technical domain, relevant data formats and expected results.
- **Input data** – The types of data accepted by the tool, including file formats, data structures or other input requirements.
- **Output data** – The types of results produced by the tool, including output formats where applicable.
- **Software type** – Whether the resource is a command-line application, Python package, notebook, workflow, web service or another type of software.
- **Version** – Information identifying the specific version.

### Writing a Good Description

The **description is particularly important**, as the Tool Registry uses semantic embeddings to support natural-language discovery and matching between datasets and tools.

A good description should answer three questions:

1. **What does the tool do?** Describe its main functionality and purpose.
2. **What kind of data does it work with?** Identify the scientific domain, input data types and relevant formats.
3. **What does it produce?** Explain the results or outputs generated by the tool.

**Example of an insufficient description:**

> A Python tool for data analysis.

**Example of a useful description:**

> A Python tool for analysing gene expression data from RNA sequencing experiments. It processes gene count matrices in CSV or TSV format, performs differential expression analysis between experimental conditions, and generates statistical results and visualisations identifying significantly expressed genes.

Providing detailed and accurate metadata improves the likelihood that the tool will be discovered when users search for tools using natural-language descriptions or submit datasets for automatic tool matching.

**Recommendation:** Include a `codemeta.json` or `ro-crate-metadata.json` file in the published repository whenever possible, even if the project already provides a `pyproject.toml`. These formats support richer descriptions of research software, particularly its scientific purpose and relationships to input and output data.

# Example git repo as a tool

In [ ]:
tool_url = {
    "url": "https://github.com/recap/DataLens.git"
}   

# Example git subfolder as a tool
Here the folder `demos/S5P` on branch `s5p` in repository `https://github.com/sebastian-luna-valero/eodc-examples` is treated as a tool 

In [ ]:
tool_url = {
    "url": "https://github.com/sebastian-luna-valero/eodc-examples/tree/s5p/demos/S5P"
}   

# Your tool URL

In [ ]:
tool_url = {
    "url": ""
}  

## Check tool registry version
This notebook was created for tool-registry version > `1.0.4`. Different versions might not have the same schema!

In [ ]:
response = requests.get(f"{API_URL_BASE}health")
if response.status_code in (200, 201):
    data = response.json()
    version = data.get("version", None)
    if version and version == TOOL_REGISTRY_VERSION:
        print(f"Correct tool registry version: {data['version']}")
    else:
        print(f"Incorrect or missing version for tool registry: {response.text}")
else:
    print(f"Failure: {response.status_code}, message: {response.text}")

## Publishing your tool
Next we can go ahead and submit the tool using a POST command.

In [ ]:
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {TOKEN}"
}
tool_id = None
response = requests.post(API_URL_SOURCES, json=tool_url, headers=headers)
if response.status_code in (200, 201):
    print("Success")
    print(response.text)
else:
    if response.status_code > 400:
        print(f"Failure: {response.status_code}")
        print(response.text)

## Check Your Tool

After submitting a tool URL to the Tool Registry, **the tool will not appear immediately** in the catalogue.

A harvesting pipeline is automatically created for the submitted URL and scheduled to run at regular intervals. During each execution, the pipeline retrieves the tool's metadata and updates the Tool Registry accordingly. This also allows changes to the tool's metadata to be detected and incorporated over time.

### Verify That Your Tool Has Been Published

Once the harvesting pipeline has successfully processed your tool, you can verify its availability by searching the Tool Registry API.

Tools can be searched by name using the following endpoint:

`GET /api/v1/tools/?name=<tool-name>`

For example, to search for a tool named **DataLens**:

[https://tool-registry-api.eosc-data-commons.dansdemo.nl/api/v1/tools/?name=DataLens](https://tool-registry-api.eosc-data-commons.dansdemo.nl/api/v1/tools/?name=DataLens)

The API returns matching tools together with their harvested metadata, allowing you to verify that the tool has been successfully registered and that its description and other metadata fields have been extracted correctly.

If your tool does not appear in the results, the harvesting pipeline may not have executed yet. Check again after the next scheduled harvesting interval.